In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
metadata_df['sample_number'] = metadata_df['sample_title'].str.replace('#', 'c').str.replace(' tumor tissue', '').str.replace(' adjancent normal tissue', '')

metadata_df = metadata_df[metadata_df['sample_title'].str.contains('tumor tissue')]

metadata_df

,sample_title,tissue,disease,sample_number
samples,,,,
GSM7099488,#1 tumor tissue,liver,hepatocellular carcinoma,c1
GSM7099490,#2 tumor tissue,liver,hepatocellular carcinoma,c2
GSM7099492,#4 tumor tissue,liver,hepatocellular carcinoma,c4
GSM7099494,#5 tumor tissue,liver,hepatocellular carcinoma,c5
GSM7099496,#8 tumor tissue,liver,hepatocellular carcinoma,c8
GSM7099498,#9 tumor tissue,liver,hepatocellular carcinoma,c9
GSM7099500,#10 tumor tissue,liver,hepatocellular carcinoma,c10
GSM7099502,#12 tumor tissue,liver,hepatocellular carcinoma,c12
GSM7099504,#13 tumor tissue,liver,hepatocellular carcinoma,c13


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined.head()

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_7666/470337480.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,disease,sample_number,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM7099488,#1 tumor tissue,liver,hepatocellular carcinoma,c1,0,2,0,0,0,0,...,2571,442,6415,2881,1944,379,7701,1601,429,1008
GSM7099490,#2 tumor tissue,liver,hepatocellular carcinoma,c2,0,1,0,0,0,0,...,2454,866,4396,968,1905,600,6712,1943,721,1460
GSM7099492,#4 tumor tissue,liver,hepatocellular carcinoma,c4,1,7,1,0,0,0,...,23961,472,9018,8832,19557,4952,21366,17123,1791,3932
GSM7099494,#5 tumor tissue,liver,hepatocellular carcinoma,c5,0,1,0,0,0,0,...,1934,423,5276,5700,1392,370,198,1261,313,754
GSM7099496,#8 tumor tissue,liver,hepatocellular carcinoma,c8,0,0,0,0,0,0,...,637,48,408,4610,302,123,678,169,159,250


In [3]:
import pandas as pd

# Download supplementary data from: https://pmc.ncbi.nlm.nih.gov/articles/instance/10083302/bin/Table2.XLSX

# Read specific sheet (sheet number 9, which is index 8)
df_clinical_data = pd.read_excel('Table 2.XLSX')
df_clinical_data.to_csv('Table_1_clinical_data.csv', index=False)

df_combined = df_combined.reset_index()
# Merge the dataframes
df_combined = df_clinical_data.merge(
    df_combined,
    left_on='sample',
    right_on='sample_number',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
# drop the Sample.ID column
df_combined = df_combined.drop(columns=['sample'])
# df_combined = df_combined.set_index('samples')
df_combined.to_csv(data_file_name, index=False)

df_combined.head()

,Etiology,Degree_of_tumor_differentiation,AFP(ng/mL),TNM_stages,BCLC,Tumor_size(mm),age,Satellites,Smoke,Alcohol,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,Cirrhosis,Poor,30783,Ⅳ,B,105,37,Present,Present,Present,...,2571,442,6415,2881,1944,379,7701,1601,429,1008
1,Hepatitis_B,Poor,2668,Ⅳ,C,190,44,Present,Absent,Absent,...,682,215,617,512,434,108,1492,392,188,525
2,Hepatitis_B,Poor,>121000,Ⅲ,C,230,43,Absent,Absent,Absent,...,48426,2209,6765,82742,21575,3843,92276,15665,11112,20309
3,Hepatitis_B,Poor,1.01,Ⅳ,C,130,64,Absent,Absent,Absent,...,681,77,519,1437,345,77,2730,296,166,423
4,Hepatitis_B,Poor,3710,Ⅳ,C,120,48,Absent,Present,Present,...,2397,415,5397,2341,1234,239,2516,1316,94,566


# Generate description


In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'Table_1_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated


In [5]:
df_combined.iloc[:, :20]

,Etiology,Degree_of_tumor_differentiation,AFP(ng/mL),TNM_stages,BCLC,Tumor_size(mm),age,Satellites,Smoke,Alcohol,PFS_day,living_day,sex,die,samples,sample_title,tissue,disease,sample_number,DDX11L1
0,Cirrhosis,Poor,30783,Ⅳ,B,105,37,Present,Present,Present,101,176,male,decensed,GSM7099488,#1 tumor tissue,liver,hepatocellular carcinoma,c1,0
1,Hepatitis_B,Poor,2668,Ⅳ,C,190,44,Present,Absent,Absent,99,190,male,decensed,GSM7099500,#10 tumor tissue,liver,hepatocellular carcinoma,c10,0
2,Hepatitis_B,Poor,>121000,Ⅲ,C,230,43,Absent,Absent,Absent,4,4,male,decensed,GSM7099502,#12 tumor tissue,liver,hepatocellular carcinoma,c12,0
3,Hepatitis_B,Poor,1.01,Ⅳ,C,130,64,Absent,Absent,Absent,224,519,female,decensed,GSM7099504,#13 tumor tissue,liver,hepatocellular carcinoma,c13,0
4,Hepatitis_B,Poor,3710,Ⅳ,C,120,48,Absent,Present,Present,108,533,male,decensed,GSM7099506,#14 tumor tissue,liver,hepatocellular carcinoma,c14,0
5,Hepatitis_B,Well,2.3,Ⅰ,A,50,66,Absent,Absent,Absent,2178,2178,male,living,GSM7099508,#15 tumor tissue,liver,hepatocellular carcinoma,c15,0
6,Hepatitis_C,Poor,54.61,Ⅱ,A,45,64,Absent,Present,Present,97,208,male,decensed,GSM7099510,#18 tumor tissue,liver,hepatocellular carcinoma,c18,1
7,Hepatitis_B,Poor,4.54,Ⅰ,A,25,40,Absent,Absent,Absent,2028,2028,male,decensed,GSM7099512,#19 tumor tissue,liver,hepatocellular carcinoma,c19,0
8,Hepatitis_B,Moderately,705.9,Ⅱ,B,75,47,Absent,Absent,Absent,367,655,female,decensed,GSM7099490,#2 tumor tissue,liver,hepatocellular carcinoma,c2,0
9,Hepatitis_B,Poor,2.41,Ⅰ,A,50,54,Absent,Absent,Absent,370,1817,male,living,GSM7099514,#20 tumor tissue,liver,hepatocellular carcinoma,c20,0
